In [2]:
import pandas as pd
import numpy as np
from football import poisson_walk_forward

columns = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR",
           "B365H", "B365D", "B365A", "MaxH", "MaxD", "MaxA"]

codes = [f"{y}{y + 1}" for y in range(19, 26)]

def load_league(league):
    seasons = []
    for code in codes:
        url = f"https://www.football-data.co.uk/mmz4281/{code}/{league}.csv"
        s = pd.read_csv(url, encoding="latin-1")
        s = s[columns].dropna(subset=["HomeTeam"])
        s["Date"] = pd.to_datetime(s["Date"], dayfirst=True)
        s["Season"] = code
        seasons.append(s)
    return pd.concat(seasons).sort_values("Date").reset_index(drop=True)

e0 = load_league("E0")
e1 = load_league("E1")
print("Premier League:", len(e0), "matches")
print("Championship:", len(e1), "matches")
print(e1.groupby("Season").size())

Premier League: 2660 matches
Championship: 3864 matches
Season
1920    552
2021    552
2122    552
2223    552
2324    552
2425    552
2526    552
dtype: int64


In [3]:
def add_predictions(df):
    seasons = sorted(df["Season"].unique())
    parts = []
    for i in range(1, len(seasons)):
        old = df[df["Season"] == seasons[i - 1]]
        new = df[df["Season"] == seasons[i]]
        parts.append(poisson_walk_forward(old, new, 2))
    pois = pd.concat(parts)
    return df.merge(pois, on=["Date", "HomeTeam", "AwayTeam"], how="inner")

e0p = add_predictions(e0)
e1p = add_predictions(e1)
print("Premier League predicted:", len(e0p))
print("Championship predicted:", len(e1p))

Premier League predicted: 2280
Championship predicted: 3312


In [4]:
def ev_backtest(df, odds_prefix, threshold=0.05):
    n_bets = 0
    profit = 0
    for outcome in ["H", "D", "A"]:
        odds = df[odds_prefix + outcome]
        ev = df["pois_" + outcome] * odds - 1
        bets = df[ev > threshold]
        won = bets["FTR"] == outcome
        profit += (bets[odds_prefix + outcome][won] - 1).sum() - (~won).sum()
        n_bets += len(bets)
    return n_bets, round(profit, 2), round(profit / n_bets * 100, 1)

for name, df in [("Premier League", e0p), ("Championship", e1p)]:
    print(name)
    print("  Bet365 odds: bets, profit, ROI% =", ev_backtest(df, "B365"))
    print("  Best odds:   bets, profit, ROI% =", ev_backtest(df, "Max"))

Premier League
  Bet365 odds: bets, profit, ROI% = (1964, np.float64(-184.36), np.float64(-9.4))
  Best odds:   bets, profit, ROI% = (2539, np.float64(-124.71), np.float64(-4.9))
Championship
  Bet365 odds: bets, profit, ROI% = (2766, np.float64(-190.56), np.float64(-6.9))
  Best odds:   bets, profit, ROI% = (3473, np.float64(-96.98), np.float64(-2.8))


In [5]:
e2p = add_predictions(load_league("E2"))
e3p = add_predictions(load_league("E3"))

leagues = [("Premier League", e0p), ("Championship", e1p),
           ("League One", e2p), ("League Two", e3p)]

for name, df in leagues:
    print(name, f"({len(df)} matches)")
    for t in [0.05, 0.10, 0.20]:
        print(f"  EV > {t:.2f}: bets, profit, ROI% =", ev_backtest(df, "Max", t))

Premier League (2280 matches)
  EV > 0.05: bets, profit, ROI% = (2539, np.float64(-124.71), np.float64(-4.9))
  EV > 0.10: bets, profit, ROI% = (2050, np.float64(-147.09), np.float64(-7.2))
  EV > 0.20: bets, profit, ROI% = (1374, np.float64(-109.17), np.float64(-7.9))
Championship (3312 matches)
  EV > 0.05: bets, profit, ROI% = (3473, np.float64(-96.98), np.float64(-2.8))
  EV > 0.10: bets, profit, ROI% = (2748, np.float64(-44.2), np.float64(-1.6))
  EV > 0.20: bets, profit, ROI% = (1761, np.float64(-51.67), np.float64(-2.9))
League One (3312 matches)
  EV > 0.05: bets, profit, ROI% = (3349, np.float64(-145.58), np.float64(-4.3))
  EV > 0.10: bets, profit, ROI% = (2684, np.float64(-161.85), np.float64(-6.0))
  EV > 0.20: bets, profit, ROI% = (1685, np.float64(-95.65), np.float64(-5.7))
League Two (3312 matches)
  EV > 0.05: bets, profit, ROI% = (3340, np.float64(-176.75), np.float64(-5.3))
  EV > 0.10: bets, profit, ROI% = (2608, np.float64(-132.21), np.float64(-5.1))
  EV > 0.20: be